# **How Transformer LLMs Work**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction](#Introduction)
  - [Course Outline](#Course-Outline)
- [Understanding Language Models: Language as a Bag-of-Words](#Understanding-Language-Models:-Language-as-a-Bag-of-Words)
  - [Understand the Landscape of Language AI Models](#Understand-the-Landscape-of-Language-AI-Models)
  - [Why Representing Language is Challenging](#Why-Representing-Language-is-Challenging)
  - [Build a Bag-of-Words Representation Step by Step](#Build-a-Bag-of-Words-Representation-Step-by-Step)
- [Understanding Language Models: (Word) Embeddings](#Understanding-Language-Models:-(Word)-Embeddings)
  - [Understand Why We Need Semantic Embeddings](#Understand-Why-We-Need-Semantic-Embeddings)
  - [Discover How Word2Vec Learns Word Meaning](#Discover-How-Word2Vec-Learns-Word-Meaning)
  - [Walk Through the Word2Vec Training Process](#Walk-Through-the-Word2Vec-Training-Process)
  - [Explore Word Embedding Dimensions and Properties](#Explore-Word-Embedding-Dimensions-and-Properties)
  - [See How Tokenization Builds Word, Sentence and Document Embeddings](#See-How-Tokenization-Builds-Word,-Sentence-and-Document-Embeddings)
- [Understanding Language Models: Encoding and Decoding Context with Attention](#Understanding-Language-Models:-Encoding-and-Decoding-Context-with-Attention)
  - [Context and RNN-Based Translation](#Context-and-RNN-Based-Translation)
  - [Walk Through the Autoregressive Translation Example](#Walk-Through-the-Autoregressive-Translation-Example)
  - [Dive Deeper into Encoding and Decoding with RNNs](#Dive-Deeper-into-Encoding-and-Decoding-with-RNNs)
  - [Understand Why Attention Improves Long-Sequence Modeling](#Understand-Why-Attention-Improves-Long-Sequence-Modeling)
  - [See How Attention Uses Hidden States for Better Outputs](#See-How-Attention-Uses-Hidden-States-for-Better-Outputs)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction

Welcome to **How Transformer LLMs Work**. In this course, you learn about the main components of the LLM transformer architecture that has transformed the field of language processing. As you leave this course, you will be able to read through papers describing models and understand the details that are used to describe these architectures. And these intuitions will help you use LLMs better too.

Let me introduce the main topic of this course the transformer. The transformer architecture was first introduced in the 2017 paper, **Attention is All You Need** by Ashish Vaswani and others for machine translation tasks. The idea was to input an English sentence and have the network output a German sentence. The same architecture tend to be great at inputting a prompt and outputting a response to that prompt like a question and the answer to that question. And so this helped herald the early rise of large language models.

The original transformer architecture consisted of two main parts: an **Encoder** and **Decoder**. Consider translating English into German, the encoder preprocess the entire input English text to extract the context needed to perform the translation. Then the decoder uses the encoder context to generate the German. The encoder and the decoder form the basis for the models used in many language models today.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-01.png" width="800">

The encoder model provides rich, context-sensitive representations of the input text, and is the basis for the BERT model and most of the embedding models used in RAG applications. The decoder model performs text generation tasks such as summarizing text, writing code, answering questions, and is the basis for most popular LLMs, such as those from OpenAI, Anthropic, Cohere, and Meta.

## Course Outline

* You first delve into recent developments in LLMs to see how a sequence of increasingly sophisticated building blocks led to the modern transformer.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-02.png" width="800">

* You then learn about tokenization, which consists of taking text and breaking it down into tokens that comprise words or word fragments. They can then be fed into the LLM.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-03.png" width="800">

* After that, you gain intuition about how the transformer network works, focusing on the decoder-only models. A generative model takes in a text prompt, and it generates a text in response by generating one token at a time.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-04.png" width="800">

    Here's how the generation process works. The model starts by mapping each input token into an embedding vector that captures the meaning of that token. After that, the model parses these token embeddings through a stack of transformer blocks, where each block is a specific neural network architecture that is designed to learn flexibly from data and also scale well on GPUs. So you learn how each block is made up of an attention layer and a feed-forward network. The model then uses the output vectors of the transformer blocks and passes them to the last component, the language modeling head, which generates the output token.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-05.png" width="800">



The magic of LLMs actually comes from two parts: One, the transformer architecture, which is well worth learning. And second, all the incredibly rich data that the models learn from. So while the magic of LLMs comes not just from the transformer architecture, but also from the data, having a solid intuition of what this architecture is doing will give you better intuitions about why they behave in certain ways, as well as how to use them.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: Language as a Bag-of-Words

## Understand the Landscape of Language AI Models

In the next several sections, you will learn the evolution of how language has been represented numerically. We'll start with **Bag of Words**, an algorithm that represents words as large sparse vectors or arrays of numbers, which simply record the presence of words. Then **Word2Vec**, whose word representation capture the meaning of words in the context of a few neighboring words. Finally, **Transformers**, whose dense vectors capture the meaning of words in the context of a sentence or a paragraph.

Although we will explore a more recent history of language AI and language models, it is important to understand where we started. Earlier techniques such as Bag-of-Words and Word2Vec have been arguably the foundation of what we're using today. Although they lack contextualized representations, they are often a good baseline to start with. We call these **non-transformer models** since today's models are typically powered solely by transformer models, in contrast to these strong baselines.

Then we have **encoder-only models**, which are great at representing language in numerical representations. In contrast, **decoder-only models** are generative in their nature. Their main usage is to generate text. And finally, we have **encoder-decoder models** that attempt to get the best of both worlds. You will learn about these non-transformer models, encoders, decoders and how they relate, as we explore this more recent history of language AI.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-06.png" width="800">

## Why Representing Language is Challenging

Language however, is a tricky concept for computers. Text is unstructured in nature and loses its meaning when represented by zeros and ones or individual characters. As a result, throughout the history of language AI, there has been a large focus on representing language in a structured manner so that it can more easily be used by computers. From generating text, to creating numerical representations, and classifying textual inputs — these are just a few of the numerous tasks you can do with language AI.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-07.png" width="800">

## Build a Bag-of-Words Representation Step by Step

At the start of this Language AI field, the focus was mainly on representing language to analyze unstructured data. A first and still very relevant method is by representing language as a bag-of-words. Imagine you have some input text, `"That is a cute dog"`. To represent this sentence you can break it up into smaller pieces. To do so, you split the text into words by separating them through a whitespace. This process of converting the input text into pieces is called *tokenization*, and each individual word is called a token. Note that a token can be any even smaller than an entire word, but we will go through tokenization in the future sections in more detail. You can perform the same tokenization process with another document, `"My cat is cute"`. Although similar documents, they do contain different words and have a different meaning. Now that you are left with two sets of tokens, you can create something called a *vocabulary*. This vocabulary contains all unique words or tokens found in both input documents. As such, the vocabulary will contain fewer words than the amount of tokens that were generated. And we typically refer to the number of tokens or words in the vocabulary as the vocabulary size.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-08.png" width="800">

To then represent one of our inputs with numerical values, let's focus a bit on our second input, `"My cat is cute"`. This input has four tokens that match with some of the words in the vocabulary, but not all. You can then count how often a certain token appears in the vocabulary that we already created. In this case, each token appears once. However, you will also have to take note of the words in the vocabulary that do not appear in the input. A sentence not only gives meaning to the words it contains, but also the words it doesn't. Now that you have counted how often the words or tokens of our input appear in the vocabulary, you have created your numerical representation. This is called a bag-of-words, and does nothing more than counting individual words that appear in the vocabulary. Therefore, the numerical representation of `"My cat is cute"` is `0101011`, in that specific order. The order is important as it allows us to compare different sentences to one another. In practice, we call this a vector representation, a list of numerical values that represents the input. In this example, these values are counts and have an explicit meaning, namely, the number of times a word in the vocabulary appears in the input. Vector representations, in more complex and advanced models, typically do not have such an intuitive meaning, and have values between 0 and 1. In the next section, we'll learn how these vector embeddings are created.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-09.png" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: (Word) Embeddings

## Understand Why We Need Semantic Embeddings

Let's explore how these more complex vector embeddings are created. Note that bag-of-words, although an elegant approach, has a flaw. It considers language to be nothing more than an almost literal bag-of-words, and ignores the semantic nature or meaning of text.

## Discover How Word2Vec Learns Word Meaning

Released in 2013, Word2Vec was one of the first successful attempts at capturing the meaning of text in embeddings. To do so, Word2Vec learned semantic representations of words by training on vast amounts of textual data, like the entirety of Wikipedia. To generate these semantic representations, Word2Vec leverages neural networks. These networks consist of interconnected layers of nodes that process information. Neural networks can have many layers, and each connection has a certain weight depending on the inputs. These weights are often referred to as parameters of the model. Using these neural networks, Word2Vec generates word embeddings by looking at which other words they tend to appear next to in a given sentence.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-10.png" width="800">

## Walk Through the Word2Vec Training Process

You start by assigning every word in your vocabulary with a vector embedding, say, five values for each word is initialized with random values. Then in every training step, you take pairs of words from training data, and the model attempts to predict whether or not they are likely to be neighbors in a sentence. During this training process, Word2Vec learns the relationship between words and distills that information into the embedding. If the two words tend to have the same neighbor, their embeddings will be closer to one another and vice versa. The resulting embeddings capture the meaning of words.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-11.png" width="800">

## Explore Word Embedding Dimensions and Properties

Embeddings capture the meaning of words. But what exactly does that mean? To illustrate this phenomenon, let's explore an example.

Assume that you have an embedding for the word `"cats"`. This embedding has values between -1 and 1. Embeddings attempt to capture meaning by representing the properties of words. For instance, the word `"cats"` might score low on the properties `newborn` and `human` and `fruits`, while scoring high on the properties `animal` and `plural`. The number of properties or values an embedding has is called the number of dimensions, and is generally a fixed size. The same can be said for other words, such as `"puppy"`, which scores high on `animal` and `newborn`, but low on all others. By doing this for a number of words, you can use these values to get a proxy of the meaning of these words. Note that the number of dimensions can be quite large, where it is not uncommon to see embeddings with more than a thousand values. However, in practice, you do not actually know what these properties exactly represent, as they are learned through complex mathematical calculations. These properties do allow you to compare embeddings and therefore words with one another. Words with similar meaning will be grouped together, whereas different words are further apart. How similar or dissimilar certain words are, depends on the training data.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-12.png" width="800">

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-13.png" width="800">

## See How Tokenization Builds Word, Sentence and Document Embeddings

Thus far we explored word embeddings, but there are many types of embeddings that we can use. When we talk about a model like Word2Vec that converts textual input to embeddings, we refer to it as a representation model as it attempts to represent text as values. Imagine that you have some input. For example, the sentence `"Her vocalization was melodic"`. Through tokenization, you can split the sentence up into tokens. Note that this procedure is actually not splitting the input by white spaces. The word `vocalization` is split up into `vocal` and `ization`. The reason for this is that models that perform tokenization, also called tokenizers, have a fixed vocabulary. As such, they cannot represent all words that exist, but sometimes have to find combinations of words. You give the representation model these individual tokens, which in turn generates these embeddings, one for each token. Note that the word `vocalization` contains the tokens, `vocal` and `ization`. So when you average the embeddings of these tokens, you get a word embedding as it now represents the entire word. Similar techniques can be used for entire sentences to create sentence embeddings, and the same for longer texts such as documents to create a document embedding. These are just a few of the many embeddings that exist.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-14.png" width="800">

In the next section, we'll learn how you can encode and decode contextualized information, rather than static representations.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: Encoding and Decoding Context with Attention

## Context and RNN-Based Translation

In the previous sections, we explored words and token embedding techniques, but with limited contextual capabilities. In this section, we will explore how you can encode and decode context with attention.

Word2Vec create static embeddings. The same embedding is generated for the word `"bank"`, regardless of the context. `"Bank"` can refer to both a financial bank as well as the bank of a river. Its meaning, and therefore its embedding should change depending on the context. Capturing the text context is important to perform some language tasks, such as translation.

A step in encoding this text was achieved through Recurrent Neural Networks or RNNs. These are variants of neural networks that can model sequences as an additional input. To do so, these RNNs are used for two tasks, encoding or representing an input sentence, and decoding, or generating an output sentence.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-15.png" width="800">

## Walk Through the Autoregressive Translation Example

Let's explore RNN-based translation by showcasing how this input sentence `"I love llamas"` gets translated to Dutch. The text is passed to the encoder, which attempts to represent the entire sequence through embeddings. The decoder then uses those embeddings to generate language. Here it translated the English `"I love llamas"` to the Dutch `"Ik hou van lama's"`.

Each step in this architecture is autoregressive. When generating the next words, this architecture needs to consume all previously generated words. For instance, you take the input `"I love llamas"`, the model then generates the first token, which is `"Ik"`. To generate the next token, the output `"Ik"` is appended to the inputs. In step two, the input is now `"I love llamas ik"` , which in turn generates the output `"hou"`. You can continue this process at each step and continuously update the inputs with the previously generated token, until the entire output is generated. Most models are autoregressive and will therefore generate a single token each time.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-16.gif" width="800">

## Dive Deeper into Encoding and Decoding with RNNs

Let's explore this encoding and decoding in a bit more detail. You again start with the input sentence `"I love llamas"` tokenized into tokens. We can use Word2Vec to create the embeddings as the inputs. Although these embeddings are static by themselves, the encoder processes the entire sequence in one go and takes into account the context of the embeddings. The encoding step aims to represent the input as well as possible, and generates the context in the form of an embedding.

The decoder in turn is in charge of generating language, and does so by leveraging the previously generated **context embedding** to eventually generate the outputs. As we explored previously, these output tokens are generated one at a time, which is called **autoregressive**. This context embedding, however, makes it difficult to deal with longer sentences, since it is merely a single embedding representing the entire input. So the single embedding might fail to capture the entire context of a long and a complex sequence.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-17.gif" width="800">

## Understand Why Attention Improves Long-Sequence Modeling

In 2014, a solution called **Attention** was introduced that highly improved upon the original architecture. Attention allows the model to focus on parts of the input sequence that are relevant to one another, or attend to each other and amplify their signal. Attention selectively determines which words are most important in a given sentence. Take our input and output sequences, for example. Words with similar meaning like `I` and the Dutch `Ik`, which are synonyms, have higher attention weights since they are more related. `I` and `llamas` have lower attention weights since they do not relate much to each other in this particular sentence. And the same applies to all other words. By adding these attention mechanisms to the decoder step, the RNN can generate signals for each input word in the sequence related to the potential outputs.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-18.png" width="800">

## See How Attention Uses Hidden States for Better Outputs

You can again represent the input using Word2Vec embeddings and pass those to the encoder. Instead of passing only a context embedding to the decoder, the hidden states of all input words are passed to the decoder. A stateful word is an internal vector from a hidden layer of an RNN that contains the information about the previous words. The decoder then uses the attention mechanism to look at the entire sequence. Finally, this again generates the language. Due to this attention mechanism, the output tends to be much better since you now look at the entire sequence using embeddings for each token or words instead of the smaller and more limited context embedding.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-19.gif" width="800">

During generation, the model attends to the most relevant inputs. After having generated `"Ik hou van"` from using the inputs `"I love llamas"`, the attention mechanism then highlights words that are more relevant in this particular example. The sequential nature of this architecture precludes parallelization during training of the model.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-20.png" width="800">

In the next section, we'll learn how attention is used in transformer models.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[DeepLearning.AI: How Transformer LLMs Work Course](https://www.deeplearning.ai/courses/how-transformer-llms-work)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)